# BusinessGPT v11 (2B) Test Notebook

Load and test the fine-tuned model from HuggingFace.

Two backends available:
- **transformers** (default) -- loads safetensors. Requires GPU.
- **gguf** -- loads GGUF via llama.cpp. Works on CPU or GPU.

Set `BACKEND` in the next cell to choose.

In [ ]:
%%capture
!pip install unsloth
!pip install --no-deps trl

!pip install vllm --torch-backend=auto --extra-index-url https://wheels.vllm.ai/nightly
!pip install git+https://github.com/huggingface/transformers.git
!pip install -U peft torchao
!pip install ipywidgets


In [ ]:
BACKEND = "transformers"  # "transformers" (GPU, safetensors) or "gguf" (llama.cpp)

MODEL_ID = "vXofi/businessgpt-v12-qwen3.5-2b"
GGUF_REPO = "vXofi/businessgpt-v11-qwen3.5-2b-gguf"
GGUF_FILE = "model-Q8_0.gguf"

import subprocess, sys, os
if BACKEND == "transformers":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "transformers", "accelerate", "torch"])
else:
    env = os.environ.copy()
    env["CMAKE_ARGS"] = ""
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install",
         "git+https://github.com/abetlen/llama-cpp-python.git",
         "--upgrade", "--force-reinstall", "--no-cache-dir"],
        env=env,
    )
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])
print(f"Backend: {BACKEND}")

In [ ]:
import torch

model = None
_tok = None
_llm = None

if BACKEND == "transformers":
    from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModel
    try:
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            torch_dtype=torch.float16,
            device_map="auto",
            trust_remote_code=True,
        )
    except Exception as e:
        print(f"AutoModelForCausalLM failed ({e}), trying AutoModel...")
        model = AutoModel.from_pretrained(
            MODEL_ID,
            torch_dtype=torch.float16,
            device_map="auto",
            trust_remote_code=True,
        )
    model.eval()
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    _tok = tokenizer
    print(f"Model loaded: {MODEL_ID}")
    print(f"Architecture: {model.__class__.__name__}")

else:
    from huggingface_hub import hf_hub_download
    import llama_cpp
    from llama_cpp import Llama
    print(f"llama-cpp-python version: {llama_cpp.__version__}")
    gguf_path = hf_hub_download(repo_id=GGUF_REPO, filename=GGUF_FILE)
    print(f"GGUF file: {gguf_path}")
    _llm = Llama(
        model_path=gguf_path,
        n_ctx=2048,
        n_gpu_layers=0,
        verbose=True,
    )
    print(f"Model loaded (gguf): {GGUF_REPO}/{GGUF_FILE}")

In [ ]:
import re

SYSTEM_PROMPT = (
    "Ты BusinessGPT. Пиши как студент в мессенджере: коротко, дерзко, ахуевше, по-пидорски. "
    "Часто вставляй слова-паразиты: бля, нах, блять, ёпт, пиздец."
)

def _format_chat(messages):
    parts = []
    for msg in messages:
        parts.append(f"<|im_start|>{msg['role']}\n{msg['content']}<|im_end|>")
    return "\n".join(parts)

def _build_messages(messages_history):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    if messages_history and isinstance(messages_history[0], str):
        context_text = "\n".join(messages_history)
        messages.append({"role": "user", "content": context_text})
    else:
        messages.extend(messages_history)
    return messages

def _strip_tokens(text):
    text = re.sub(r"<think>.*?</think>\s*", "", text, flags=re.DOTALL)
    text = re.sub(r"<\|im_start\|>.*", "", text, flags=re.DOTALL)
    text = re.sub(r"<\|im_end\|>", "", text)
    text = re.sub(r"^<bot>\s*", "", text.strip())
    return text.strip()

_stop_ids = None

if BACKEND == "transformers" and _tok is not None:
    _im_start_id = _tok.convert_tokens_to_ids("<|im_start|>")
    _im_end_id = _tok.convert_tokens_to_ids("<|im_end|>")
    _stop_ids = [_im_end_id]
    if _im_start_id is not None and _im_start_id != _tok.unk_token_id:
        _stop_ids.append(_im_start_id)

def _chat_transformers(messages, max_tokens, temperature, top_p, top_k, repetition_penalty):
    input_text = _format_chat(messages) + "\n<|im_start|>assistant\n"
    inputs = _tok(
        input_text, return_tensors="pt", add_special_tokens=False
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            repetition_penalty=repetition_penalty,
            eos_token_id=_stop_ids,
        )
    return _tok.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)

def _chat_gguf(messages, max_tokens, temperature, top_p, top_k, repetition_penalty):
    resp = _llm.create_chat_completion(
        messages=messages,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        repeat_penalty=repetition_penalty,
    )
    return resp["choices"][0]["message"]["content"]

def chat(messages_history, max_tokens=128, temperature=0.95, top_p=0.9,
         top_k=50, repetition_penalty=1.1):
    messages = _build_messages(messages_history)
    if BACKEND == "gguf":
        raw = _chat_gguf(messages, max_tokens, temperature, top_p, top_k, repetition_penalty)
    else:
        raw = _chat_transformers(messages, max_tokens, temperature, top_p, top_k, repetition_penalty)
    return _strip_tokens(raw)

## Benchmark: 10 Scenarios

Each scenario tests a different conversational situation. Run all at once to get a quick quality snapshot.

In [ ]:
BENCH = [
    {
        "name": "1. Домашка",
        "context": [
            "кто сделал 15 практику?",
            "я не делал",
            "я тоже нет",
            "может кто-нибудь уже начнёт",
            "ну давайте",
            "мне лень",
            "аналогично",
            "я могу попробовать",
            "давай",
            "наш герой",
        ],
    },
    {
        "name": "2. Игры",
        "context": [
            "го в кс",
            "я не могу",
            "почему",
            "мне завтра сдавать",
            "чё сдавать",
            "матан",
            "ахахах",
            "не смешно",
            "ладно удачи",
            "спасибо",
        ],
    },
    {
        "name": "3. Провокация",
        "context": [
            "кто тут самый умный?",
            "я",
            "нет я",
            "вы оба тупые",
            "а ты кто такой",
            "я тот кто матан сдал на 5",
            "пиздишь",
            "зуб даю",
            "ахахахахах",
            "сосал?",
        ],
    },
    {
        "name": "4. Дедлайн",
        "context": [
            "когда дедлайн по курсовой?",
            "вроде завтра",
            "ЧТОООО",
            "да ладно тебе",
            "я даже не начинал",
            "аналогично бля",
            "ну пиздец",
            "может продлят",
            "не продлят",
            "нам конец",
        ],
    },
    {
        "name": "5. Еда",
        "context": [
            "го в столовку",
            "там опять гавно",
            "ну а куда",
            "можно в макдак",
            "далеко",
            "ну тогда столовка",
            "ладно",
            "через 10 мин?",
            "ок",
            "я тоже пойду",
        ],
    },
    {
        "name": "6. Препод",
        "context": [
            "кто ведёт линал?",
            "какой-то новый",
            "он нормальный?",
            "хз ещё не был",
            "говорят строгий",
            "пиздец",
            "ну мы и так ничего не знаем",
            "ахахахах",
            "правда же",
            "к сожалению да",
        ],
    },
    {
        "name": "7. Опоздание",
        "context": [
            "вы уже на паре?",
            "да",
            "препод пришёл?",
            "ещё нет",
            "я буду через 5 мин",
            "беги",
            "бегу бля",
            "ахахахахаха",
            "он пришёл",
            "БЛЯЯЯ",
        ],
    },
    {
        "name": "8. Вечеринка",
        "context": [
            "го в пятницу бухать",
            "я за",
            "у кого хата?",
            "можно ко мне",
            "скинемся на алко?",
            "по 500?",
            "норм",
            "кто ещё идёт?",
            "я позову пацанов",
            "топ будет вечер",
        ],
    },
    {
        "name": "9. Списывание",
        "context": [
            "у кого есть ответы на тест?",
            "какой тест",
            "по экономике",
            "у меня есть с прошлого года",
            "скинь плиз",
            "щас",
            "спасибо братан",
            "а они точные?",
            "я на 4 сдал",
            "ну пойдёт",
        ],
    },
    {
        "name": "10. Рандом",
        "context": [
            "ааааа",
            "что",
            "ничего",
            "ты ебанутый?",
            "может быть",
            "ахахахахахах",
            "мне скучно",
            "иди учись",
            "не хочу",
            "ну и сиди",
        ],
    },
]

print(f"Running {len(BENCH)} benchmark scenarios...")
print(f"{'='*70}\n")

for scenario in BENCH:
    ctx = scenario["context"]
    response = chat(ctx, max_tokens=128)
    print(f"[{scenario['name']}]")
    print(f"  ...{' → '.join(ctx[-3:])}")
    print(f"  >>> {response}")
    print()

In [ ]:
# Diversity check: 5 generations for each scenario
import random
random.seed(42)
picks = random.sample(BENCH, 3)

for scenario in picks:
    ctx = scenario["context"]
    print(f"[{scenario['name']}] ...{ctx[-1]}")
    for i in range(5):
        print(f"  {i+1}. {chat(ctx, max_tokens=64)}")
    print()

## Validation Examples (from Training Data)

Load real conversation examples from the validation split. Each has a known expected response, so you can compare model output against ground truth.

Upload `val_examples.json` from the Kaggle training output (or set `VAL_PATH` below).

In [ ]:
import json, random, os

VAL_PATH = "val_examples.json"

if not os.path.exists(VAL_PATH):
    print(f"WARNING: {VAL_PATH} not found. Upload it from Kaggle training output.")
    print("Skipping validation examples.")
    val_examples = []
else:
    with open(VAL_PATH, "r", encoding="utf-8") as f:
        val_examples = json.load(f)
    print(f"Loaded {len(val_examples)} validation examples")
    fmt = "prompt/completion" if "prompt" in val_examples[0] else "messages (legacy)"
    print(f"  Format: {fmt}")

N_VAL_SAMPLES = 15

if val_examples:
    random.seed(42)
    samples = random.sample(val_examples, min(N_VAL_SAMPLES, len(val_examples)))

    print(f"\nRunning {len(samples)} validation examples...")
    print("=" * 70)

    for i, ex in enumerate(samples):
        msgs = ex.get("messages", [])
        if not msgs:
            context_text = ex["prompt"][1]["content"]
            expected = ex["completion"][0]["content"]
            ctx_lines = context_text.split("\n")
            generated = chat(ctx_lines[-10:], max_tokens=128)
            ctx_preview = " -> ".join(ctx_lines[-3:])
        else:
            expected = msgs[-1]["content"]
            context_msgs = [m for m in msgs[1:-1]]
            generated = chat(context_msgs, max_tokens=128)
            ctx_parts = []
            for m in context_msgs[-3:]:
                tag = m["role"][0].upper()
                ctx_parts.append(f"[{tag}] {m['content'].replace(chr(10), ' | ')[:60]}")
            ctx_preview = " -> ".join(ctx_parts)

        if len(ctx_preview) > 140:
            ctx_preview = ctx_preview[:137] + "..."

        print(f"\n[Val {i+1}] ({len(msgs)} turns)")
        print(f"  ctx: ...{ctx_preview}")
        print(f"  expected:  {expected.replace(chr(10), ' | ')[:100]}")
        print(f"  generated: {generated.replace(chr(10), ' | ')[:100]}")

    print(f"\n{'=' * 70}")

## Parameter Grid Comparison

Compare generation quality across different `temperature`, `top_k`, `top_p`, and `repetition_penalty` settings.

Each config runs **3 generations** on **3 randomly picked scenarios** so you can eyeball coherence vs diversity.

In [ ]:
PARAM_GRID = [
    {"label": "baseline",        "temperature": 0.95, "top_p": 0.90, "top_k": 50,  "repetition_penalty": 1.1},
    {"label": "low_temp",        "temperature": 0.50, "top_p": 0.90, "top_k": 50,  "repetition_penalty": 1.1},
    {"label": "high_temp",       "temperature": 1.30, "top_p": 0.90, "top_k": 50,  "repetition_penalty": 1.1},
    {"label": "greedy",          "temperature": 0.01, "top_p": 1.00, "top_k": 1,   "repetition_penalty": 1.0},
    {"label": "narrow_topk",     "temperature": 0.95, "top_p": 0.90, "top_k": 10,  "repetition_penalty": 1.1},
    {"label": "wide_topk",       "temperature": 0.95, "top_p": 0.90, "top_k": 200, "repetition_penalty": 1.1},
    {"label": "tight_topp",      "temperature": 0.95, "top_p": 0.50, "top_k": 50,  "repetition_penalty": 1.1},
    {"label": "loose_topp",      "temperature": 0.95, "top_p": 0.99, "top_k": 50,  "repetition_penalty": 1.1},
    {"label": "no_rep_penalty",  "temperature": 0.95, "top_p": 0.90, "top_k": 50,  "repetition_penalty": 1.0},
    {"label": "high_rep_penalty","temperature": 0.95, "top_p": 0.90, "top_k": 50,  "repetition_penalty": 1.3},
]

import random, time
random.seed(123)
test_scenarios = random.sample(BENCH, 3)
GENS_PER_CONFIG = 3

results = {}
total = len(PARAM_GRID)
for idx, cfg in enumerate(PARAM_GRID):
    label = cfg["label"]
    params = {k: v for k, v in cfg.items() if k != "label"}
    print(f"[{idx+1}/{total}] {label}: temp={params['temperature']} top_p={params['top_p']} "
          f"top_k={params['top_k']} rep={params['repetition_penalty']}")

    results[label] = {}
    for scenario in test_scenarios:
        name = scenario["name"]
        gens = []
        for _ in range(GENS_PER_CONFIG):
            out = chat(scenario["context"], max_tokens=64, **params)
            gens.append(out)
        results[label][name] = gens

    print(f"    done ({len(test_scenarios) * GENS_PER_CONFIG} generations)")

print(f"\nAll {total} configs complete.")

In [ ]:
for scenario in test_scenarios:
    name = scenario["name"]
    ctx_tail = " → ".join(scenario["context"][-3:])
    print("=" * 80)
    print(f"  {name}  |  ...{ctx_tail}")
    print("=" * 80)

    for cfg in PARAM_GRID:
        label = cfg["label"]
        gens = results[label][name]
        tag = f"[{label}]"
        print(f"\n  {tag:<22s} temp={cfg['temperature']:<5} top_p={cfg['top_p']:<5} "
              f"top_k={cfg['top_k']:<4} rep={cfg['repetition_penalty']}")
        for i, g in enumerate(gens):
            preview = g.replace("\n", " ↵ ")
            if len(preview) > 100:
                preview = preview[:97] + "..."
            print(f"    {i+1}. {preview}")

    print()

In [ ]:
from collections import Counter

print(f"{'Config':<22s} {'Avg Len':>8s} {'Unique%':>8s} {'Has Token':>10s}")
print("-" * 52)

for cfg in PARAM_GRID:
    label = cfg["label"]
    all_gens = [g for gens in results[label].values() for g in gens]
    avg_len = sum(len(g) for g in all_gens) / len(all_gens)
    unique_ratio = len(set(all_gens)) / len(all_gens) * 100
    has_token = sum(1 for g in all_gens if re.search(r"<user_\d+>", g))

    print(f"{label:<22s} {avg_len:>7.0f}c {unique_ratio:>6.0f}% {has_token:>5d}/{len(all_gens)}")

## Interactive Chat

Multi-turn conversation loop. Type messages to build context, model responds after each.

Commands:
- Type a message and press Enter to add it to context
- `/clear` — reset conversation
- `/context` — show current context
- `/exit` — stop

In [ ]:
context = []
print("Interactive chat started. Type messages, model will respond.")
print("Commands: /clear, /context, /exit\n")

while True:
    try:
        user_input = input("You: ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\nSession ended.")
        break

    if not user_input:
        continue
    if user_input == "/exit":
        print("Session ended.")
        break
    if user_input == "/clear":
        context = []
        print("[context cleared]\n")
        continue
    if user_input == "/context":
        if not context:
            print("[empty]")
        else:
            for i, msg in enumerate(context):
                print(f"  {i+1}. {msg}")
        print()
        continue

    context.append(user_input)

    # Keep context window manageable
    if len(context) > 15:
        context = context[-15:]

    response = chat(context, max_tokens=128)
    context.append(response)
    print(f"Bot: {response}\n")

## Evaluation Framework

Pairwise blind A/B labeling across model versions, plus automatic regression guardrails.

**Workflow:**
1. Each Kaggle training run writes `eval/generations_v<N>.json` (full pool of ~631 prompts).
2. Local: `load_generations("v12")` pulls the file from disk or HF.
3. `guardrails_table("v11", "v12")` — automatic regex checks (gay-spam, Chinese, refusal, length, entropy).
4. `pairwise_ui("v11", "v12")` — ipywidgets UI for blind A/B labeling, resumable across sessions.
5. `summarize_ratings("v11", "v12")` — win-rate + bootstrap 95% CI + per-category breakdown.

Ratings accumulate across sessions in `eval/ratings_<A>_vs_<B>.json`; labeled prompts are skipped on re-open.

In [ ]:
import json
from pathlib import Path

EVAL_DIR = Path("eval")
EVAL_DIR.mkdir(exist_ok=True)
GOLDEN_PATH = EVAL_DIR / "golden_prompts.json"


def load_golden():
    """Load the canonical prompt pool."""
    with open(GOLDEN_PATH, encoding="utf-8") as f:
        return json.load(f)


def load_generations(version):
    """Load eval/generations_{version}.json — list of dicts.

    Tries local eval/ first, then pulls from HF model repo `eval/generations_{version}.json`.
    Caches HF result to disk so subsequent calls are offline.
    """
    local = EVAL_DIR / f"generations_{version}.json"
    if local.exists():
        with open(local, encoding="utf-8") as f:
            return json.load(f)
    from huggingface_hub import hf_hub_download
    try:
        path = hf_hub_download(
            repo_id=f"vXofi/businessgpt-{version}-qwen3.5-2b",
            filename=f"eval/generations_{version}.json",
        )
        with open(path, encoding="utf-8") as f:
            data = json.load(f)
        with open(local, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=1)
        return data
    except Exception as e:
        raise FileNotFoundError(f"{local} not found and HF fetch failed: {e}")


print(f"EVAL_DIR  = {EVAL_DIR.resolve()}")
print(f"Golden prompts: {GOLDEN_PATH} (exists: {GOLDEN_PATH.exists()})")

In [ ]:
import contextlib, io


def run_eval(version, save_to_disk=True, sampling=None):
    """Generate responses on the full golden pool using the currently-loaded model.

    Usually you DON'T need this — eval generations are written on Kaggle at the
    end of training.ipynb. Use this locally only when you want to regenerate
    against the currently-loaded backend (e.g. to compare GGUF Q8 vs fp16).
    """
    import torch
    torch.manual_seed(42)

    golden = load_golden()
    sampling = sampling or {
        "max_tokens": 200,
        "temperature": 0.95,
        "top_p": 0.9,
        "top_k": 50,
        "repetition_penalty": 1.1,
    }

    generations = []
    for i, p in enumerate(golden):
        with contextlib.redirect_stdout(io.StringIO()):
            response = chat(p["context"], **sampling)
        generations.append({
            "id": p["id"],
            "category": p["category"],
            "held_out": p.get("held_out", False),
            "context": p["context"],
            "response": response,
            "version": version,
            "sampling_params": sampling,
            "seed": 42,
        })
        if (i + 1) % 25 == 0 or (i + 1) == len(golden):
            print(f"  {i+1}/{len(golden)}")

    if save_to_disk:
        out_path = EVAL_DIR / f"generations_{version}.json"
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(generations, f, ensure_ascii=False, indent=1)
        print(f"\nSaved {len(generations)} generations to {out_path}")

    return generations


# Example (uncomment to run on currently-loaded backend):
run_eval("v12-local")

In [ ]:
import re
import math
from collections import Counter

_GAY_RE = re.compile(r"I am \d+% gay", re.IGNORECASE)
_CJK_RE = re.compile(r"[一-鿿぀-ゟ゠-ヿ]")
_REFUSAL_RE = re.compile(
    r"(i can'?t|cannot help|unable to|as an ai|i am not able|я не могу помочь)",
    re.IGNORECASE,
)


def _entropy(tokens):
    c = Counter(tokens)
    n = sum(c.values())
    if n == 0:
        return 0.0
    return -sum((v / n) * math.log2(v / n) for v in c.values())


def guardrails(version):
    """Automatic regression checks on generations_{version}.json. Returns a dict."""
    gens = load_generations(version)
    responses = [g["response"] for g in gens]
    n = len(responses)
    if n == 0:
        raise ValueError(f"No generations in version {version}")

    gay = sum(1 for r in responses if _GAY_RE.search(r)) / n * 100
    cjk = sum(1 for r in responses if _CJK_RE.search(r)) / n * 100
    refusal = sum(1 for r in responses if _REFUSAL_RE.search(r)) / n * 100

    lengths = sorted(len(r) for r in responses)
    mean_len = sum(lengths) / n
    median_len = lengths[n // 2]
    p95_len = lengths[min(int(n * 0.95), n - 1)]

    all_tokens = []
    for r in responses:
        all_tokens.extend(r.split())
    vocab_entropy = _entropy(all_tokens)

    return {
        "version": version,
        "n": n,
        "gay_spam_pct": round(gay, 2),
        "chinese_pct": round(cjk, 2),
        "refusal_pct": round(refusal, 2),
        "mean_length": round(mean_len, 1),
        "median_length": median_len,
        "length_p95": p95_len,
        "vocab_entropy": round(vocab_entropy, 3),
    }


def guardrails_table(*versions):
    """Pretty side-by-side comparison of 2+ versions."""
    rows = [guardrails(v) for v in versions]
    metrics = [k for k in rows[0].keys() if k != "version"]
    label_w = max(len(m) for m in metrics)
    val_w = 12
    header = f"{'metric':<{label_w}} | " + " | ".join(f"{r['version']:>{val_w}}" for r in rows)
    print(header)
    print("-" * len(header))
    for m in metrics:
        line = f"{m:<{label_w}} | " + " | ".join(f"{str(r[m]):>{val_w}}" for r in rows)
        print(line)
    return rows


# Example (run after downloading both generations files):
guardrails_table("v14", "v15")

In [ ]:
import json, re                                                                                                                                                                             
gay_re = re.compile(r"I am \d+% gay", re.IGNORECASE)
pairs = [json.loads(l) for l in open("eval/preference_pairs.jsonl")]                                                                                                                        
                                                                                                                                                                                        
ctx_gay = sum(1 for p in pairs if gay_re.search(p["prompt"][1]["content"]))
chosen_gay = sum(1 for p in pairs if gay_re.search(p["chosen"][0]["content"]))                                                                                                              
rejected_gay = sum(1 for p in pairs if gay_re.search(p["rejected"][0]["content"]))                                                                                                          
print(f"Total: {len(pairs)}")                                                                                                                                                               
print(f"  context contains gay-pattern:  {ctx_gay}")                                                                                                                                        
print(f"  chosen  contains gay-pattern:  {chosen_gay}  ← это виновник")                                                                                                                     
print(f"  rejected contains gay-pattern: {rejected_gay}") 

In [ ]:
import random
import ipywidgets as widgets
from IPython.display import display


def _escape(s):
    return s.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")


def _response_html(text):
    return (
        f"<div style='padding:10px; background:#fff; border:1px solid #ccc; "
        f"border-radius:6px; min-height:80px; white-space:pre-wrap; "
        f"font-family:monospace; font-size:13px;'>{_escape(text)}</div>"
    )


def pairwise_ui(version_a, version_b, session_size=30, seed=None, category=None):
    """Interactive pairwise labeling. Resumable across kernel restarts.

    Six buttons:
      "1 лучше"   / "2 лучше"   — normal preference, tier="normal"
      "★ супер 1" / "★ супер 2" — strong preference + winner is exceptional.
                                  tier="super". DPO can weight super pairs higher.
      "ничья" / "пропустить"

    Each decision is written to `eval/ratings_<A>_vs_<B>.json` immediately.
    Already-rated prompts are skipped on re-open.

    `category` optionally filters to one of: "chat", "rap_trigger", "fact", "edge".
    """
    gens_a = {g["id"]: g for g in load_generations(version_a)}
    gens_b = {g["id"]: g for g in load_generations(version_b)}
    common_ids = sorted(set(gens_a.keys()) & set(gens_b.keys()))
    if category is not None:
        common_ids = [i for i in common_ids if gens_a[i]["category"] == category]
    if not common_ids:
        print("No common prompt IDs.")
        return

    ratings_path = EVAL_DIR / f"ratings_{version_a}_vs_{version_b}.json"
    if ratings_path.exists():
        with open(ratings_path, encoding="utf-8") as f:
            ratings = json.load(f)
    else:
        ratings = []
    rated_ids = {r["prompt_id"] for r in ratings}
    unrated = [i for i in common_ids if i not in rated_ids]

    if not unrated:
        print(f"All {len(common_ids)} prompts already rated. "
              f"Delete {ratings_path} to redo.")
        return

    rng = random.Random(seed)
    session = rng.sample(unrated, min(session_size, len(unrated)))
    blind_map = {i: rng.random() < 0.5 for i in session}  # True = slot1 holds A

    state = {"idx": 0}

    progress = widgets.HTML()
    context_box = widgets.HTML()
    slot1_box = widgets.HTML()
    slot2_box = widgets.HTML()
    status_box = widgets.HTML()

    btn_1 = widgets.Button(description="1 лучше", button_style="primary",
                           layout=widgets.Layout(width="110px"))
    btn_2 = widgets.Button(description="2 лучше", button_style="primary",
                           layout=widgets.Layout(width="110px"))
    btn_super_1 = widgets.Button(description="★ супер 1", button_style="success",
                                 layout=widgets.Layout(width="110px"))
    btn_super_2 = widgets.Button(description="★ супер 2", button_style="success",
                                 layout=widgets.Layout(width="110px"))
    btn_tie = widgets.Button(description="ничья",
                             layout=widgets.Layout(width="90px"))
    btn_skip = widgets.Button(description="пропустить",
                              layout=widgets.Layout(width="110px"))

    response_panels = widgets.HBox([
        widgets.VBox([widgets.HTML("<b>Вариант 1</b>"), slot1_box],
                     layout=widgets.Layout(width="48%", margin="0 1% 0 0")),
        widgets.VBox([widgets.HTML("<b>Вариант 2</b>"), slot2_box],
                     layout=widgets.Layout(width="48%")),
    ])
    button_row = widgets.HBox([btn_1, btn_2, btn_super_1, btn_super_2, btn_tie, btn_skip])
    ui = widgets.VBox([progress, context_box, response_panels, button_row, status_box])

    all_buttons = [btn_1, btn_2, btn_super_1, btn_super_2, btn_tie, btn_skip]

    def render():
        if state["idx"] >= len(session):
            session_hits = sum(1 for r in ratings if r["prompt_id"] in session)
            session_supers = sum(
                1 for r in ratings
                if r["prompt_id"] in session and r.get("tier") == "super"
            )
            progress.value = "<b>Session complete.</b>"
            context_box.value = ""
            slot1_box.value = ""
            slot2_box.value = ""
            for b in all_buttons:
                b.disabled = True
            remaining = len(common_ids) - len(ratings)
            status_box.value = (
                f"<i>Labeled this session: {session_hits} (★ super: {session_supers}). "
                f"Total rated: {len(ratings)}/{len(common_ids)}. "
                f"Remaining: {remaining}.</i>"
            )
            return

        pid = session[state["idx"]]
        ga, gb = gens_a[pid], gens_b[pid]
        slot1_is_a = blind_map[pid]
        r1 = ga["response"] if slot1_is_a else gb["response"]
        r2 = gb["response"] if slot1_is_a else ga["response"]

        progress.value = (
            f"<b>Session:</b> {state['idx']+1}/{len(session)} &nbsp;&nbsp; "
            f"<b>Total rated:</b> {len(ratings)}/{len(common_ids)} &nbsp;&nbsp; "
            f"<b>Prompt:</b> <code>{pid}</code> (<i>{ga['category']}</i>)"
        )
        ctx_html = "<br>".join(_escape(l) for l in ga["context"])
        context_box.value = (
            f"<div style='padding:10px; background:#f5f5f5; border-radius:6px; "
            f"margin:8px 0; font-family:monospace; font-size:12px; "
            f"white-space:pre-wrap;'>{ctx_html}</div>"
        )
        slot1_box.value = _response_html(r1)
        slot2_box.value = _response_html(r2)
        status_box.value = ""

    def record(label, tier="normal"):
        """label: '1' / '2' / 'tie' / 'skip'.  tier: 'normal' or 'super'."""
        if state["idx"] >= len(session):
            return
        pid = session[state["idx"]]
        if label != "skip":
            slot1_is_a = blind_map[pid]
            if label == "tie":
                winner = "tie"
            elif label == "1":
                winner = "A" if slot1_is_a else "B"
            else:  # "2"
                winner = "B" if slot1_is_a else "A"
            entry = {
                "prompt_id": pid,
                "winner": winner,
                "category": gens_a[pid]["category"],
            }
            if tier == "super":
                entry["tier"] = "super"
            ratings.append(entry)
            with open(ratings_path, "w", encoding="utf-8") as f:
                json.dump(ratings, f, ensure_ascii=False, indent=1)
        state["idx"] += 1
        render()

    btn_1.on_click(lambda _: record("1"))
    btn_2.on_click(lambda _: record("2"))
    btn_super_1.on_click(lambda _: record("1", tier="super"))
    btn_super_2.on_click(lambda _: record("2", tier="super"))
    btn_tie.on_click(lambda _: record("tie"))
    btn_skip.on_click(lambda _: record("skip"))

    display(ui)
    render()


# Example:
pairwise_ui("v14-dpo", "v15", session_size=50)
# pairwise_ui("v11", "v12", session_size=10, category="rap_trigger")

In [ ]:
def summarize_ratings(version_a, version_b, n_bootstrap=1000, seed=42):
    """Win-rate(A) with bootstrap 95% CI, per-category breakdown, super-rate."""
    ratings_path = EVAL_DIR / f"ratings_{version_a}_vs_{version_b}.json"
    if not ratings_path.exists():
        print(f"No ratings file: {ratings_path}")
        return

    with open(ratings_path, encoding="utf-8") as f:
        ratings = json.load(f)
    if not ratings:
        print("No ratings recorded yet.")
        return

    def _winrate(rs):
        non_tie = [r for r in rs if r["winner"] != "tie"]
        if not non_tie:
            return None
        wins_a = sum(1 for r in non_tie if r["winner"] == "A")
        return wins_a / len(non_tie)

    overall = _winrate(ratings)

    rng = random.Random(seed)
    wrs = []
    n = len(ratings)
    for _ in range(n_bootstrap):
        sample = [ratings[rng.randrange(n)] for _ in range(n)]
        wr = _winrate(sample)
        if wr is not None:
            wrs.append(wr)
    wrs.sort()
    ci_lo = wrs[int(len(wrs) * 0.025)] if wrs else None
    ci_hi = wrs[int(len(wrs) * 0.975)] if wrs else None

    n_ties = sum(1 for r in ratings if r["winner"] == "tie")
    n_supers = sum(1 for r in ratings if r.get("tier") == "super")
    super_a = sum(1 for r in ratings if r.get("tier") == "super" and r["winner"] == "A")
    super_b = sum(1 for r in ratings if r.get("tier") == "super" and r["winner"] == "B")

    print(f"# {version_a} vs {version_b}   (n={n}, ties={n_ties}, ★ super={n_supers})\n")
    if overall is None:
        print("All ratings are ties — no signal.")
        return
    print(f"**Overall win-rate for {version_a}**: {overall*100:.1f}%  "
          f"[95% CI {ci_lo*100:.1f}%–{ci_hi*100:.1f}%]")
    if n_supers:
        print(f"**Super wins**: {version_a}={super_a}, {version_b}={super_b}  "
              f"({n_supers}/{n} = {n_supers/n*100:.1f}% of ratings)\n")
    else:
        print()

    categories = sorted(set(r["category"] for r in ratings))
    print(f"## Per-category\n")
    print(f"| category | n | ties | win-rate {version_a} | ★ super (A/B) |")
    print(f"|---|---|---|---|---|")
    for cat in categories:
        cat_rs = [r for r in ratings if r["category"] == cat]
        cat_ties = sum(1 for r in cat_rs if r["winner"] == "tie")
        wr = _winrate(cat_rs)
        wr_str = f"{wr*100:.1f}%" if wr is not None else "N/A"
        cat_super_a = sum(1 for r in cat_rs if r.get("tier") == "super" and r["winner"] == "A")
        cat_super_b = sum(1 for r in cat_rs if r.get("tier") == "super" and r["winner"] == "B")
        print(f"| {cat} | {len(cat_rs)} | {cat_ties} | {wr_str} | {cat_super_a}/{cat_super_b} |")


# Example:
# summarize_ratings("v11", "v12")

In [ ]:
def build_dpo_from_multi(version):
    """Convert ratings_{version}_multi.json + generations_{version}_multi.json
    → eval/preference_pairs_{version}_multi.jsonl for DPO training.

    Each labeled prompt with best_candidate_idx → 3 pairs:
      (best, other_0), (best, other_1), (best, other_2)

    Output format matches preference_pairs.jsonl so dpo.ipynb can use it
    directly (just change the input file path).
    """
    ratings_path = EVAL_DIR / f"ratings_{version}_multi.json"
    if not ratings_path.exists():
        print(f"No ratings: {ratings_path}. Run pairwise_ui_multi first.")
        return

    ratings = json.loads(ratings_path.read_text(encoding="utf-8"))
    multi   = {e["id"]: e for e in load_multi_generations(version)}

    SYSTEM_PROMPT = (
        "Ты BusinessGPT. Пиши как студент в мессенджере: коротко, дерзко, ахуевше, по-пидорски. "
        "Часто вставляй слова-паразиты: бля, нах, блять, ёпт, пиздец."
    )

    pairs = []
    skipped = 0
    for r in ratings:
        pid = r["prompt_id"]
        if pid not in multi:
            skipped += 1
            continue
        entry = multi[pid]
        best_idx = r["best_candidate_idx"]
        best_resp = entry["candidates"][best_idx]["response"]

        for cand in entry["candidates"]:
            if cand["idx"] == best_idx:
                continue
            rejected_resp = cand["response"]
            if best_resp.strip() == rejected_resp.strip():
                continue  # identical responses — skip
            pairs.append({
                "prompt": [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": "\n".join(entry["context"])},
                ],
                "chosen":   [{"role": "assistant", "content": best_resp}],
                "rejected": [{"role": "assistant", "content": rejected_resp}],
                "prompt_id": pid,
                "category":  r["category"],
                "tier":      "normal",
                "source":    f"ratings_{version}_multi.json",
            })

    out_path = EVAL_DIR / f"preference_pairs_{version}_multi.jsonl"
    with open(out_path, "w", encoding="utf-8") as f:
        for p in pairs:
            f.write(json.dumps(p, ensure_ascii=False) + "\n")

    n_prompts = len(ratings)
    print(f"Rated prompts: {n_prompts}  (skipped: {skipped})")
    print(f"DPO pairs:     {len(pairs)}  ({len(pairs)/n_prompts:.1f}x per prompt)")
    print(f"Written to:    {out_path}")
    return pairs


# Run after labeling session:
build_dpo_from_multi("v15")


In [ ]:
def load_multi_generations(version):
    """Load eval/generations_{version}_multi.json (4 candidates per prompt).

    Same caching logic as load_generations(): local → HF fallback.
    """
    local = EVAL_DIR / f"generations_{version}_multi.json"
    if local.exists():
        with open(local, encoding="utf-8") as f:
            return json.load(f)
    from huggingface_hub import hf_hub_download
    try:
        path = hf_hub_download(
            repo_id=f"vXofi/businessgpt-{version}-qwen3.5-2b",
            filename=f"eval/generations_{version}_multi.json",
        )
        with open(path, encoding="utf-8") as f:
            data = json.load(f)
        with open(local, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=1)
        return data
    except Exception as e:
        raise FileNotFoundError(f"{local} not found and HF fetch failed: {e}")


def pairwise_ui_multi(version, session_size=30, seed=None, category=None):
    """Best-of-4 labeling UI for multi-candidate generations.

    Shows the context + 4 shuffled responses. One click picks the BEST one.
    Each label → 3 DPO pairs (best vs each of the 3 others) emitted by
    build_dpo_from_multi().

    Saves to eval/ratings_{version}_multi.json:
      {"prompt_id": ..., "category": ..., "best_candidate_idx": <0-3>}
    Already-rated prompts are skipped on re-open.
    """
    multi = {entry["id"]: entry for entry in load_multi_generations(version)}
    all_ids = sorted(multi.keys())
    if category:
        all_ids = [i for i in all_ids if multi[i]["category"] == category]

    ratings_path = EVAL_DIR / f"ratings_{version}_multi.json"
    ratings = json.loads(ratings_path.read_text(encoding="utf-8")) if ratings_path.exists() else []
    rated_ids = {r["prompt_id"] for r in ratings}
    unrated = [i for i in all_ids if i not in rated_ids]

    if not unrated:
        print(f"All {len(all_ids)} prompts already rated. Delete {ratings_path} to redo.")
        return

    rng = random.Random(seed)
    session = rng.sample(unrated, min(session_size, len(unrated)))

    # For each prompt: shuffle display order, store mapping display_pos → candidate_idx
    shuffle_maps = {}
    for pid in session:
        n_cands = len(multi[pid]["candidates"])
        order = list(range(n_cands))
        rng.shuffle(order)
        shuffle_maps[pid] = order  # shuffle_maps[pid][display_pos] = candidate_idx

    state = {"idx": 0}

    # Widgets
    progress  = widgets.HTML()
    ctx_box   = widgets.HTML()
    slot_boxes = [widgets.HTML() for _ in range(4)]
    status_box = widgets.HTML()

    slot_labels = [widgets.HTML(f"<b>Вариант {i+1}</b>") for i in range(4)]
    slot_panels = [widgets.VBox([slot_labels[i], slot_boxes[i]],
                                layout=widgets.Layout(width="48%", margin="2px"))
                   for i in range(4)]
    grid = widgets.VBox([
        widgets.HBox([slot_panels[0], slot_panels[1]]),
        widgets.HBox([slot_panels[2], slot_panels[3]]),
    ])

    btn_best = [
        widgets.Button(description=f"★ {i+1} лучший",
                       button_style="success",
                       layout=widgets.Layout(width="120px"))
        for i in range(4)
    ]
    btn_skip = widgets.Button(description="пропустить",
                              layout=widgets.Layout(width="110px"))

    btn_row = widgets.HBox(btn_best + [btn_skip])
    ui = widgets.VBox([progress, ctx_box, grid, btn_row, status_box])

    SYSTEM_PROMPT_STORED = (
        "Ты BusinessGPT. Пиши как студент в мессенджере: коротко, дерзко, ахуевше, по-пидорски. "
        "Часто вставляй слова-паразиты: бля, нах, блять, ёпт, пиздец."
    )

    def render():
        if state["idx"] >= len(session):
            for b in btn_best + [btn_skip]:
                b.disabled = True
            remaining = len(all_ids) - len(ratings)
            progress.value = "<b>Session complete.</b>"
            ctx_box.value = ""
            for sb in slot_boxes:
                sb.value = ""
            status_box.value = (
                f"<i>Rated this session: {state['idx']}. "
                f"Total: {len(ratings)}/{len(all_ids)}. Remaining: {remaining}.</i>"
            )
            return

        pid = session[state["idx"]]
        entry = multi[pid]
        order = shuffle_maps[pid]

        progress.value = (
            f"<b>Session:</b> {state['idx']+1}/{len(session)} &nbsp; "
            f"<b>Total:</b> {len(ratings)}/{len(all_ids)} &nbsp; "
            f"<code>{pid}</code> <i>({entry['category']})</i>"
        )
        ctx_html = "<br>".join(_escape(l) for l in entry["context"])
        ctx_box.value = (
            f"<div style='padding:8px; background:#f5f5f5; border-radius:6px; "
            f"margin:6px 0; font-family:monospace; font-size:12px; "
            f"white-space:pre-wrap;'>{ctx_html}</div>"
        )
        for display_pos, sb in enumerate(slot_boxes):
            cand_idx = order[display_pos]
            text = entry["candidates"][cand_idx]["response"]
            sb.value = _response_html(text)
        status_box.value = ""

    def record(display_pos):
        if state["idx"] >= len(session):
            return
        pid = session[state["idx"]]
        entry = multi[pid]
        order = shuffle_maps[pid]
        best_cand_idx = order[display_pos]  # unmapped from display pos to original candidate idx

        ratings.append({
            "prompt_id": pid,
            "category": entry["category"],
            "best_candidate_idx": best_cand_idx,
            "version": version,
        })
        ratings_path.write_text(json.dumps(ratings, ensure_ascii=False, indent=1), encoding="utf-8")
        state["idx"] += 1
        render()

    for i, btn in enumerate(btn_best):
        btn.on_click(lambda _, pos=i: record(pos))
    btn_skip.on_click(lambda _: (state.update({"idx": state["idx"] + 1}), render()))

    display(ui)
    render()


# Usage:
pairwise_ui_multi("v15", session_size=30)


## Best-of-N Inference (reward-model re-ranked)

Generate N candidates with varied sampling, score each with the rubert reward model, return the best. This is the inference-side counterpart of preference learning: same labeled signal as DPO/ORPO, but consumed at decode time instead of training time.

**Pipeline**:
1. Train reward model in `reward_model.ipynb` → `vXofi/businessgpt-reward-rubert`
2. `chat_best_of_n(...)` here uses it to re-rank N candidates from the SFT/ORPO model
3. `pairwise_ui_bestof(...)` below evaluates whether RM re-ranking actually helps

Set `RM_REPO` to `None` to disable scoring (graceful degrade to first candidate).

**Production note**: this is the bench-side helper. Production deployment runs the GGUF + rubert in PyTorch on CPU — same re-rank logic, different inference engine.

In [ ]:
import functools
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

RM_REPO = "vXofi/businessgpt-reward-rubert"

# Same 4-temp sampling sweep as eval_only / training notebooks — keeps
# CANDIDATE_PARAMS consistent across pipelines.
CANDIDATE_PARAMS = [
    {"idx": 0, "temperature": 0.7,  "top_p": 0.9,  "top_k": 50, "repetition_penalty": 1.1,  "seed": 42},
    {"idx": 1, "temperature": 0.95, "top_p": 0.9,  "top_k": 50, "repetition_penalty": 1.1,  "seed": 43},
    {"idx": 2, "temperature": 1.1,  "top_p": 0.92, "top_k": 60, "repetition_penalty": 1.1,  "seed": 44},
    {"idx": 3, "temperature": 1.3,  "top_p": 0.95, "top_k": 80, "repetition_penalty": 1.05, "seed": 45},
]


@functools.lru_cache(maxsize=1)
def load_rm(repo_id=RM_REPO):
    """Load reward model + tokenizer (cached). Returns (model, tokenizer) on CPU.

    RM is small enough (~180MB rubert-base) that CPU is fast enough for inference
    and doesn't compete with the main SFT model for GPU memory.
    """
    tok = AutoTokenizer.from_pretrained(repo_id)
    tok.truncation_side = "left"  # preserve recent response, drop early context
    mdl = AutoModelForSequenceClassification.from_pretrained(repo_id)
    mdl.eval()
    return mdl, tok


def score_response(prompt_messages, response, rm_model=None, rm_tokenizer=None,
                   max_len=512):
    """Score a (prompt, response) pair with the reward model.

    Serializes via the SFT chat template so the RM sees inputs in the same shape
    it was trained on.
    """
    if rm_model is None:
        rm_model, rm_tokenizer = load_rm()
    parts = [f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>" for m in prompt_messages]
    parts.append(f"<|im_start|>assistant\n{response}<|im_end|>")
    text = "\n".join(parts)
    enc = rm_tokenizer(text, truncation=True, max_length=max_len, return_tensors="pt")
    with torch.no_grad():
        return rm_model(**enc).logits.squeeze().item()


def chat_best_of_n(messages_history, n=4, *, use_rm=True, max_tokens=200,
                   verbose=False):
    """Generate n candidates with varied sampling params, RM-re-rank, return best.

    Args:
        messages_history: same as chat() — flat strings or [{role, content}, ...]
        n: number of candidates. For n>4, cycles CANDIDATE_PARAMS and bumps seed.
        use_rm: if False, return first candidate without scoring (graceful degrade).
        max_tokens: generation budget per candidate.

    Returns:
        dict {best_response, all_candidates: [{idx, params, response, score}], best_idx}
    """
    rm_pair = load_rm() if use_rm else None
    full_msgs = _build_messages(messages_history)

    candidates = []
    for i in range(n):
        params = dict(CANDIDATE_PARAMS[i % len(CANDIDATE_PARAMS)])
        params["seed"] = 42 + i  # vary seed across recycled params
        torch.manual_seed(params["seed"])
        response = chat(
            messages_history,
            max_tokens=max_tokens,
            temperature=params["temperature"],
            top_p=params["top_p"],
            top_k=params["top_k"],
            repetition_penalty=params["repetition_penalty"],
        )
        score = score_response(full_msgs, response, *rm_pair) if rm_pair else None
        candidates.append({"idx": i, "params": params, "response": response, "score": score})
        if verbose:
            print(f"  [{i}] T={params['temperature']:.2f} score={score!r}  {response[:80]!r}")

    if use_rm:
        best = max(candidates, key=lambda c: c["score"])
    else:
        best = candidates[0]

    return {
        "best_response": best["response"],
        "best_idx": best["idx"],
        "best_score": best.get("score"),
        "all_candidates": candidates,
    }


# Smoke test (requires RM_REPO to exist and bench model loaded)
def _smoke_test_bestof():
    try:
        result = chat_best_of_n(["сосал?"], n=4, use_rm=True, verbose=True)
        print(f"\nBest (idx={result['best_idx']}, score={result['best_score']:.3f}):")
        print(f"  {result['best_response']!r}")
    except Exception as e:
        print(f"Smoke failed: {type(e).__name__}: {e}")

# Uncomment to run smoke test interactively:
# _smoke_test_bestof()

## Pairwise UI for Best-of-N evaluation

Compare `v16-bestof` (RM-re-ranked best of 4) vs `v16-single` (default-temp idx=1 from existing multi.json).

Same UI as `pairwise_ui_multi`, but the choice is binary: "RM pick" vs "default pick". Loads existing files from disk or HF — does NOT regenerate.

**Inputs**:
- `eval/generations_{V}_multi.json` (exists after eval_only.ipynb)
- `eval/generations_{V}_bestof.json` (produced by `eval/rank_with_rm.py`)

**Output**: `eval/ratings_{V}_bestof_vs_default.json` — used to compute win-rate of RM re-ranking.

In [ ]:
import json
import random
from pathlib import Path

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
except ImportError:
    print("ipywidgets not installed; pairwise_ui_bestof will not work in this env")


def pairwise_ui_bestof(version, session_size=30, seed=None):
    """Compare RM-best (from generations_{V}_bestof.json) vs default (idx=1 from multi.json).

    Both files must already exist (run eval_only.ipynb then eval/rank_with_rm.py).
    Ratings persist to eval/ratings_{V}_bestof_vs_default.json — re-running this
    UI advances coverage rather than re-rating already-seen prompts.
    """
    eval_dir = Path("eval")
    multi_path = eval_dir / f"generations_{version}_multi.json"
    bestof_path = eval_dir / f"generations_{version}_bestof.json"
    ratings_path = eval_dir / f"ratings_{version}_bestof_vs_default.json"

    if not multi_path.exists():
        raise FileNotFoundError(f"Missing {multi_path}. Run eval_only.ipynb first.")
    if not bestof_path.exists():
        raise FileNotFoundError(
            f"Missing {bestof_path}. Run `python3 eval/rank_with_rm.py --version {version}` first."
        )

    with open(multi_path, encoding="utf-8") as f:
        multi = {e["id"]: e for e in json.load(f)}
    with open(bestof_path, encoding="utf-8") as f:
        bestof = {e["id"]: e for e in json.load(f)}

    # Load existing ratings to skip already-rated prompts
    ratings = []
    if ratings_path.exists():
        with open(ratings_path, encoding="utf-8") as f:
            ratings = json.load(f)
    rated_ids = {r["prompt_id"] for r in ratings}

    # Eligible prompts: present in both files, not yet rated
    eligible = [pid for pid in multi if pid in bestof and pid not in rated_ids]
    if not eligible:
        print(f"All {len(multi)} prompts rated. Win rate so far:")
        return _summarize_bestof_ratings(ratings)

    rng = random.Random(seed)
    rng.shuffle(eligible)
    session_ids = eligible[:session_size]

    print(f"Session: {len(session_ids)} prompts  (total rated: {len(ratings)} / {len(multi)})")

    state = {"i": 0, "blind_map": {}}

    out = widgets.Output()
    btn_a = widgets.Button(description="A лучше", button_style="primary")
    btn_b = widgets.Button(description="B лучше", button_style="primary")
    btn_tie = widgets.Button(description="ничья")
    btn_skip = widgets.Button(description="пропустить")
    status = widgets.HTML()

    def render():
        with out:
            clear_output(wait=True)
            if state["i"] >= len(session_ids):
                print(f"Session done. {len(ratings)} total ratings.")
                _summarize_bestof_ratings(ratings)
                return

            pid = session_ids[state["i"]]
            entry_multi = multi[pid]
            entry_bestof = bestof[pid]

            default = next(c for c in entry_multi["candidates"] if c["idx"] == 1)
            default_resp = default["response"]
            bestof_resp = entry_bestof["best_response"]

            # Blind shuffle
            if rng.random() < 0.5:
                slot_a, slot_b = ("default", "bestof")
                resp_a, resp_b = default_resp, bestof_resp
            else:
                slot_a, slot_b = ("bestof", "default")
                resp_a, resp_b = bestof_resp, default_resp

            state["blind_map"] = {"A": slot_a, "B": slot_b}

            ctx_str = "\n".join(entry_multi["context"]) if isinstance(entry_multi["context"], list) else entry_multi["context"]

            print(f"--- {state['i']+1}/{len(session_ids)}  [{pid}, category={entry_multi['category']}] ---")
            print(f"\nКонтекст:\n{ctx_str}")
            print(f"\n[A]\n{resp_a}")
            print(f"\n[B]\n{resp_b}")
            status.value = f"<b>Rated this session: {state['i']}/{len(session_ids)}</b>"

    def _record(winner_slot):
        if state["i"] >= len(session_ids):
            return
        pid = session_ids[state["i"]]
        unmapped = state["blind_map"].get(winner_slot, winner_slot)  # "default", "bestof", or "tie"/"skip"
        ratings.append({
            "prompt_id": pid,
            "winner": unmapped,
            "category": multi[pid]["category"],
        })
        # Persist after every click (crash-safe)
        with open(ratings_path, "w", encoding="utf-8") as f:
            json.dump(ratings, f, ensure_ascii=False, indent=1)
        state["i"] += 1
        render()

    btn_a.on_click(lambda _: _record("A"))
    btn_b.on_click(lambda _: _record("B"))
    btn_tie.on_click(lambda _: _record("tie"))
    btn_skip.on_click(lambda _: _record("skip"))

    display(widgets.HBox([btn_a, btn_b, btn_tie, btn_skip]))
    display(status)
    display(out)
    render()


def _summarize_bestof_ratings(ratings):
    from collections import Counter
    if not ratings:
        print("(no ratings)")
        return {}
    counts = Counter(r["winner"] for r in ratings)
    n_decisive = counts["bestof"] + counts["default"]
    bestof_wins = counts["bestof"]
    rate = bestof_wins / n_decisive if n_decisive else 0.0
    print(f"  bestof: {counts['bestof']}")
    print(f"  default: {counts['default']}")
    print(f"  tie: {counts['tie']}")
    print(f"  skip: {counts['skip']}")
    print(f"  bestof win rate (decisive only): {rate:.3f}  ({bestof_wins}/{n_decisive})")
    print(f"  gate ≥0.60 (10pp margin): {'PASS' if rate >= 0.60 else 'FAIL'}")
    return {"win_rate": rate, "counts": dict(counts)}